# Fitting & Training Pipeline

This notebook takes the dataset and uses it to train 3 models, each tuned with cross-validated
hyperparameter search, explicit overfitting controls, and an explicit final-training step:

1. Random Forest
2. SVM
3. XGBoost

## Responsibilities of this notebook
- Load the pre-split train/test data, the `StratifiedKFold` splitter, and make the preprocessing
  template produced by an earlier notebook.
- For each model: run randomized hyperparameter search with k-fold CV (regularization-focused
  search spaces, plus early stopping for XGBoost) to find the best hyperparameters.
- Explicitly (re)train each model's final pipeline on the **full** training set using those
  winning hyperparameters.
- Evaluate each final model once on the held-out test set.
- Persist the fitted pipelines, full CV results, and test-set metrics under `../Models/` and
  `../Results/` so they can be reloaded and analyzed later without rerunning training.

## When using Google colab

In [1]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Google Drive mounted successfully.")
except ImportError:
    IN_COLAB = False
    print("Not running in Google Colab environment. Using local directory paths.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.


In [2]:
if IN_COLAB:
    !pip install -q xgboost "scikit-learn>=1.9.0" pandas pyarrow fastparquet joblib


## 1. Imports and configuration

In [3]:
# ------------------------- imports -------------------------
import json, os, joblib
import numpy as np
import pandas as pd
from scipy.stats import loguniform

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, StandardScaler, OneHotEncoder
from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV,
    train_test_split,
    cross_val_predict,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    brier_score_loss,
    log_loss,
    confusion_matrix,
    classification_report,
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [4]:
# ------------------------- Paths -------------------------
if IN_COLAB:
    PROJECT_ROOT = "/content/drive/MyDrive/Projects/Week1and2Project"
else:
    PROJECT_ROOT = ".."

print(f"Using PROJECT_ROOT: {os.path.abspath(PROJECT_ROOT)}")

# Directory paths based on PROJECT_ROOT
ARTIFACTS_DIR = os.path.join(PROJECT_ROOT, "Artifacts")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "Results")
MODELS_DIR = os.path.join(PROJECT_ROOT, "Models")
DATAFRAMES_DIR = os.path.join(PROJECT_ROOT, "DataFrames")
CV_RESULTS_DIR = os.path.join(RESULTS_DIR, "cv_results")
METRICS_DIR = os.path.join(RESULTS_DIR, "metrics")

for d in (ARTIFACTS_DIR, RESULTS_DIR, MODELS_DIR, CV_RESULTS_DIR, METRICS_DIR, DATAFRAMES_DIR):
    os.makedirs(d, exist_ok=True)
print("All project directories confirmed and ready.")

Using PROJECT_ROOT: /content/drive/MyDrive/Projects/Week1and2Project
All project directories confirmed and ready.


In [5]:
# ------------------------- Var -------------------------
TARGET = "Churn"
RANDOM_STATE = 42
N_JOBS = -1

N_ITER_SEARCH = 40          # 'RandomizedSearchCV' candidates per model (SVM uses fewer)
EARLY_STOPPING_ROUNDS = 30  # XGBoost only
ES_VALIDATION_SIZE = 0.15   # fraction of X_train carved out to monitor early stopping

## 2. Load all required data

In [6]:
# Loading Data Splits
X_train = pd.read_parquet(os.path.join(DATAFRAMES_DIR, 'X_train.parquet'))
y_train = pd.read_parquet(os.path.join(DATAFRAMES_DIR, 'y_train.parquet'))

X_test = pd.read_parquet(os.path.join(DATAFRAMES_DIR, 'X_test.parquet'))
y_test = pd.read_parquet(os.path.join(DATAFRAMES_DIR, 'y_test.parquet'))

In [7]:
# Flaten y to 1-D array/Series
def _to_1d_target(y):
    if isinstance(y, pd.DataFrame):
        y = y[TARGET] if TARGET in y.columns else y.iloc[:, 0]
    y = y.squeeze()
    if not pd.api.types.is_numeric_dtype(y):
        y = y.map({"No": 0, "Yes": 1, "0": 0, "1": 1, False: 0, True: 1}).astype(int)
    return y

y_train = _to_1d_target(y_train)
y_test = _to_1d_target(y_test)

print(f"X_train Length: {len(X_train)}, y_train Length: {len(y_train)}")
print(f"X_test Length: {len(X_test)}, y_test Length: {len(y_test)}")
print("\nTrain class balance:")
print(y_train.value_counts(normalize=True))

X_train Length: 5616, y_train Length: 5616
X_test Length: 1405, y_test Length: 1405

Train class balance:
Churn
0    0.735577
1    0.264423
Name: proportion, dtype: float64


In [8]:
# preprocessor = joblib.load(os.path.join(ARTIFACTS_DIR, 'preprocessor_template.pkl'))
skf = joblib.load(os.path.join(ARTIFACTS_DIR, 'skf.pkl'))

print("skf:", skf)

skf: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


## 3. Preprocessing pipelines

The important idea is **fit preprocessing inside the model pipeline**.

- Numeric features: median imputation.
- Categorical features: most-frequent imputation + one-hot encoding.
- Scaling is added only where it matters (Logistic Regression and SVM).

Tree models do not need standardized numeric values.

In [9]:
numeric_cols = ['tenure', 'SecuritySupportCount', 'StreamingCount',
 'TotalCharges', 'MonthlyCharges']

categorical_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
 'PaymentMethod','HasFamily', 'IsNewCustomer']

In [10]:
def make_preprocessor(scale_numeric=False):
    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipe = Pipeline(numeric_steps)
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", drop="if_binary"))
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_cols),
        ("categorical", categorical_pipe, categorical_cols),
    ])

tree_preprocessor = make_preprocessor(scale_numeric=False)
scaled_preprocessor = make_preprocessor(scale_numeric=True)

## 3. Tuning / final-training / evaluation / persistence helpers

Three separate helpers, one per responsibility, so the notebook makes it obvious *when* and *on
what data* each thing happens:

- **`search_hyperparameters`**:

  runs `RandomizedSearchCV` (on the shared K-folds) to find
  the best hyperparameters. `refit="roc_auc"` is required by sklearn for `best_params_` /
  `best_score_` to be populated at all, but the pipeline that internal refit produces is
  discarded, we never touch `search.best_estimator_`. The real final-training step happens
  explicitly.

---------------------------------------------------------------------------------------------------
- **`train_final_model`**:

  clones the untouched pipeline template, applies the winning
  hyperparameters, and explicitly calls `.fit(X_train, y_train)` on the **entire** training set.
  This is the actual final-model-training step.

---------------------------------------------------------------------------------------------------
- **`evaluate_and_save`**:
  scores the already-fitted final pipeline once on the held-out test
  set, then persists the pipeline + metrics to `../Models/` and `../Results/`.

In [11]:
def search_hyperparameters(name, pipeline, param_distributions, n_iter=N_ITER_SEARCH):
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring={
            "roc_auc": "roc_auc",
            "average_precision": "average_precision",
            "f1": "f1"
        },
        refit="roc_auc",
        cv=skf,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=1,
        return_train_score=True,
    )
    search.fit(X_train, y_train)

    best_idx = search.best_index_
    cv_rows.append({
        "Model": name,
        "CV ROC-AUC": search.cv_results_["mean_test_roc_auc"][best_idx],
        "CV AP": search.cv_results_["mean_test_average_precision"][best_idx],
        "CV F1": search.cv_results_["mean_test_f1"][best_idx],
        "Best Params": search.best_params_
    })

    print("Best CV ROC-AUC:", f"{search.best_score_:.4f}")
    print("Best params:", search.best_params_)

    cv_results_df = pd.DataFrame(cv_rows).sort_values("CV ROC-AUC", ascending=False).set_index("Model")
    display(cv_results_df[["CV ROC-AUC", "CV AP", "CV F1"]])
    return search

In [12]:
def train_final_model(pipeline_template, best_params):
    final_pipeline = clone(pipeline_template).set_params(**best_params)
    final_pipeline.fit(X_train, y_train)
    return final_pipeline

In [13]:
def evaluate_and_save(name, fitted_pipeline, threshold, extra_info=None):
    y_pred = fitted_pipeline.predict(X_test)
    y_proba = fitted_pipeline.predict_proba(X_test)[:, 1]

    predictions = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    test_metrics = {
        "Model": name,
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "Average Precision": average_precision_score(y_test, y_proba),
        "Accuracy": accuracy_score(y_test, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "Specificity": specificity,
        "F1": f1_score(y_test, predictions, zero_division=0),
        "F2": fbeta_score(y_test, predictions, beta=2, zero_division=0),
        "MCC": matthews_corrcoef(y_test, predictions),
        "Brier": brier_score_loss(y_test, y_proba),
        "Log Loss": log_loss(y_test, y_proba),
        "Threshold": threshold,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
    }

    if extra_info:
        test_metrics.update(extra_info)

    # joblib.dump(fitted_pipeline, os.path.join(MODELS_DIR, f"{name}_best_pipeline.pkl"))
    # with open(os.path.join(METRICS_DIR, f"{name}_test_metrics.json"), "w") as f:
    #     json.dump(test_metrics, f, indent=2, default=str)

    print(f"\n=== {name.upper()} — held-out test performance ===")
    print(classification_report(y_test, y_pred, zero_division=0))
    print(f"ROC AUC: {test_metrics['ROC-AUC']:.4f}")

    return test_metrics


### Freeze model selection and create out-of-fold probabilities

**This is the key improvement over the old notebook.**

We use out-of-fold (OOF) predictions from the training set to choose a classification threshold. The final test set remains untouched.

This matters because `0.50` is only a default threshold; it is not automatically the best threshold for F1 or recall.

In [14]:
def best_threshold(y_true, probabilities, beta=1.0):
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = []
    for t in thresholds:
        pred = (probabilities >= t).astype(int)
        scores.append(fbeta_score(y_true, pred, beta=beta, zero_division=0))
    idx = int(np.argmax(scores))
    return float(thresholds[idx]), float(scores[idx])


def oof(name, search):
    oof_probs = cross_val_predict(
        clone(search.best_estimator_),
        X_train,
        y_train,
        cv=skf,
        method="predict_proba",
        n_jobs=N_JOBS
    )[:, 1]
    threshold, score = best_threshold(y_train.to_numpy(), oof_probs, beta=1.0)

    print(f"{name}: OOF max-F1 threshold={threshold:.3f}, OOF F1={score:.4f}")
    return oof_probs, threshold


# Training the KFold-tuned models

In [15]:
cv_rows = []
oof_probabilities = {}
test_rows = []


## 1. Random Forest

In [16]:
rf_param_distributions = {
    "model__n_estimators": [150, 200, 300, 400, 500],
    "model__max_depth": [4, 6, 8, 10, 12, 16, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", 0.5, 0.8],
    "model__class_weight": ["balanced", "balanced_subsample"],
}

rf_pipeline = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=1,
    )),
])

rf_search = search_hyperparameters("random_forest", rf_pipeline, rf_param_distributions)


Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best CV ROC-AUC: 0.8482
Best params: {'model__n_estimators': 200, 'model__min_samples_split': 2, 'model__min_samples_leaf': 8, 'model__max_features': 'log2', 'model__max_depth': None, 'model__class_weight': 'balanced'}


,CV ROC-AUC,CV AP,CV F1
Model,,,
random_forest,0.848186,0.661035,0.635348


Getting the best threshold

In [17]:
rf_oof, rf_threshold = oof('Random Forest', rf_search)
oof_probabilities["Random Forest"] = rf_oof


Random Forest: OOF max-F1 threshold=0.470, OOF F1=0.6371


*Train the final Random Forest on the full training set*

In [18]:
rf_final_pipeline = train_final_model(rf_pipeline, rf_search.best_params_)
joblib.dump(rf_final_pipeline, os.path.join(MODELS_DIR, 'rf_final_model.joblib'))

rf_test_metrics = evaluate_and_save(
    "Random Forest",
    rf_final_pipeline,
    threshold=rf_threshold,
    extra_info={"best_cv_roc_auc": rf_search.best_score_, "best_params": rf_search.best_params_},
)
test_rows.append(rf_test_metrics)



=== RANDOM FOREST — held-out test performance ===
              precision    recall  f1-score   support

           0       0.90      0.75      0.82      1033
           1       0.53      0.77      0.63       372

    accuracy                           0.76      1405
   macro avg       0.72      0.76      0.73      1405
weighted avg       0.80      0.76      0.77      1405

ROC AUC: 0.8408


## 2. SVM

In [19]:
svc_param_distributions = {
    "model__C": loguniform(1e-2, 1e2),
    "model__gamma": loguniform(1e-4, 1e0),
}

svc_pipeline = Pipeline([
    ("preprocessor", scaled_preprocessor),
    ("model", SVC(
        kernel="rbf",
        probability=True,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    )),
])

svc_search = search_hyperparameters("svm", svc_pipeline, svc_param_distributions,
n_iter=N_ITER_SEARCH)


Fitting 5 folds for each of 40 candidates, totalling 200 fits


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Best CV ROC-AUC: 0.8446
Best params: {'model__C': np.float64(21.368329072358772), 'model__gamma': np.float64(0.0007068974950624604)}


,CV ROC-AUC,CV AP,CV F1
Model,,,
random_forest,0.848186,0.661035,0.635348
svm,0.844598,0.659811,0.598524


Getting the best threshold

In [20]:
svc_oof, svc_threshold = oof('SVM', svc_search)
oof_probabilities["SVC"] = svc_oof


SVM: OOF max-F1 threshold=0.465, OOF F1=0.6367


*Train the final SVM on the full training set*

In [ ]:
svc_final_pipeline = train_final_model(svc_pipeline, svc_search.best_params_)
joblib.dump(svc_final_pipeline, os.path.join(MODELS_DIR, 'svc_final_model.joblib'))

svc_test_metrics = evaluate_and_save(
    "SVC",
    svc_final_pipeline,
    threshold=svc_threshold,
    extra_info={"best_cv_roc_auc": svc_search.best_score_, "best_params": svc_search.best_params_},
)
test_rows.append(svc_test_metrics)


## 3. XGBoost

In [22]:
xgb_param_distributions = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__learning_rate": loguniform(1e-2, 3e-1),
    "model__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5, 7],
    "model__gamma": [0, 0.1, 0.3, 0.5, 1.0],
    "model__reg_alpha": [0, 0.001, 0.01, 0.1, 1, 5],     # L1
    "model__reg_lambda": [0.1, 1, 2, 5, 10, 20],          # L2
}

neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print(f"scale_pos_weight (neg/pos): {scale_pos_weight:.3f}")

xgb_pipeline = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("model", XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=1,
        scale_pos_weight=scale_pos_weight,
    )),
])

xgb_search = search_hyperparameters("xgboost", xgb_pipeline, xgb_param_distributions)


scale_pos_weight (neg/pos): 2.782
Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best CV ROC-AUC: 0.8504
Best params: {'model__colsample_bytree': 0.6, 'model__gamma': 0.5, 'model__learning_rate': np.float64(0.02713423743251271), 'model__max_depth': 5, 'model__min_child_weight': 5, 'model__n_estimators': 100, 'model__reg_alpha': 0.01, 'model__reg_lambda': 2, 'model__subsample': 0.6}


,CV ROC-AUC,CV AP,CV F1
Model,,,
xgboost,0.850374,0.668461,0.639636
random_forest,0.848186,0.661035,0.635348
svm,0.844598,0.659811,0.598524


### Early stopping refinement

`search_hyperparameters` above picked the best structural/regularization hyperparameters, but
every candidate was trained for a fixed `n_estimators`. Here we reuse those winning
hyperparameters and let early stopping choose the actual number of boosting rounds on a held-out
validation slice of the training data, so the final model stops as soon as validation loss stops
improving instead of continuing to fit noise. The test set (`X_test` / `y_test`) is never
touched during this step.

In [ ]:
best_xgb_params = {k.replace("model__", ""): v for k, v in xgb_search.best_params_.items()}
best_xgb_params.pop("n_estimators", None)  # will be superseded by the early-stopped value

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=ES_VALIDATION_SIZE, random_state=RANDOM_STATE, stratify=y_train
)

es_preprocessor = clone(tree_preprocessor).fit(X_tr)
X_tr_enc = es_preprocessor.transform(X_tr)
X_val_enc = es_preprocessor.transform(X_val)

es_model = XGBClassifier(
    **best_xgb_params,
    n_estimators=2000,   # high ceiling; early stopping will cut this short
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    n_jobs=-1,
)
es_model.fit(X_tr_enc, y_tr, eval_set=[(X_val_enc, y_val)], verbose=False)

best_n_estimators = es_model.best_iteration + 1
print(f"Early stopping selected {best_n_estimators} boosting rounds (cap was 2000).")


Getting the best threshold

In [ ]:
xgb_oof, xgb_threshold = oof('XGBoost', xgb_search)
oof_probabilities["XGBoost"] = xgb_oof


### Train the final XGBoost model on the full training set

In [25]:
final_xgb_params = dict(best_xgb_params)
final_xgb_params["n_estimators"] = best_n_estimators  # picked by early stopping, above

final_xgb_pipeline = Pipeline([
    ("preprocessor", clone(tree_preprocessor)),
    ("model", XGBClassifier(
        **final_xgb_params,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
    )),
])
final_xgb_pipeline.fit(X_train, y_train)  # explicit final training on the FULL training set
joblib.dump(final_xgb_pipeline, os.path.join(MODELS_DIR, 'xgboost_final_model.joblib'))


['/content/drive/MyDrive/Projects/Week1and2Project/Models/xgboost_final_model.joblib']

In [26]:
xgb_test_metrics = evaluate_and_save(
    "XGBoost",
    final_xgb_pipeline,
    threshold=xgb_threshold,
    extra_info={
        "best_params_from_search": xgb_search.best_params_,
        "best_cv_roc_auc": xgb_search.best_score_,
        "early_stopping_best_n_estimators": best_n_estimators,
    },
)
test_rows.append(xgb_test_metrics)



=== XGBOOST — held-out test performance ===
              precision    recall  f1-score   support

           0       0.90      0.76      0.82      1033
           1       0.53      0.78      0.63       372

    accuracy                           0.76      1405
   macro avg       0.72      0.77      0.73      1405
weighted avg       0.81      0.76      0.77      1405

ROC AUC: 0.8426


# Model comparison & results summary

In [27]:
oof_df = pd.DataFrame(oof_probabilities, index=X_train.index).sort_index()
oof_df["Churn"] = y_train
oof_df.to_parquet(os.path.join(RESULTS_DIR, "oof_probabilities.parquet"))

cv_results_df = pd.DataFrame(cv_rows).sort_values("CV ROC-AUC", ascending=False).set_index("Model")
cv_results_df.reset_index().to_json(os.path.join(CV_RESULTS_DIR, "model_cv_results.json"), orient="records", indent=2)

test_results = pd.DataFrame(test_rows)
test_results.reset_index().to_csv(os.path.join(METRICS_DIR, "model_test_results.csv"), index=False)


In [28]:
summary_rows = []
for model_name, metrics in [
    ("Random Forest", rf_test_metrics),
    ("SVC", svc_test_metrics),
    ("XGBoost", xgb_test_metrics),
]:
    summary_rows.append({
        "model": model_name,
        "test_roc_auc": metrics["ROC-AUC"],
        "test_precision": metrics["Precision"],
        "test_recall": metrics["Recall"],
        "test_f1": metrics["F1"],
        "test_accuracy": metrics["Accuracy"],
        "best_cv_roc_auc": metrics.get("best_cv_roc_auc"),
    })

pd.DataFrame(summary_rows).sort_values("test_roc_auc", ascending=False).reset_index(drop=True)


,model,test_roc_auc,test_precision,test_recall,test_f1,test_accuracy,best_cv_roc_auc
0,XGBoost,0.842620,0.535448,0.771505,0.632159,0.762278,0.850374
1,Random Forest,0.840808,0.510274,0.801075,0.623431,0.743772,0.848186
2,SVC,0.836546,0.521898,0.768817,0.621739,0.752313,0.844598
